# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Lane: Lane 2 — Refresh / Content Opportunity Scoring** (continuing from w01/w02). w01/w02 worked
off the 30k-row starter CSV — one row was one static page. The warehouse below is a *daily* fact
table, not a snapshot, so the unit of analysis and the proxy target both get re-derived for that
finer grain in sections 1–2, rather than assuming the CSV's `is_declining_label` still applies.

In [ ]:
# Setup: clone the repo (for skills/ and requirements), connect DuckDB to the warehouse.
import os, sys, subprocess

REPO_URL = "https://github.com/abuhussein1504/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

%pip -q install duckdb
import duckdb

con = duckdb.connect()
# In Colab: from google.colab import userdata; token = userdata.get('HF_TOKEN')
from google.colab import userdata
token = userdata.get('HF_TOKEN')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{token}')")

MONTH = "month=2026-03"  # mid-panel month — never the sealed final-month _sample table
FACT = f"read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/{MONTH}/*.parquet')"
DIM_CONTENT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"
DIM_CLIENTS = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')"
print("connected, working dir:", os.getcwd())

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

---

**One row = one content item, on one report_date** ("content-day") — a single client's page as
it looked in Search Console/Analytics that day. This is the grain of
`fact_content_daily_performance`; the query below checks it (grouping by `content_id,
report_date` and counting should return zero rows with count > 1).

**Time window:** the `month=2026-03` partition, `2026-03-01` through `2026-03-31` — a mid-panel
month, developed on deliberately, never the `_sample` table (that's the sealed final month, June
2026). Any feature I build later uses only rows with `report_date <= d` for a decision day `d`.

---

In [ ]:
# Query A — grain probe. Zero rows back confirms one row really is one content-day.
grain_probe = con.sql(f"""
    SELECT content_id, report_date, COUNT(*) AS c
    FROM {FACT}
    GROUP BY content_id, report_date
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
grain_probe  # expect: empty dataframe

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

---

| Field | Bucket | Why |
|---|---|---|
| `clicks`, `impressions`, `gsc_avg_position` (trailing, up to `d`) | Feature | knowable before the decision moment |
| `ga4_data_available` (trailing, up to `d`) | Feature | flags whether GA4 values are real vs placeholder — itself knowable at `d` |
| `click_decline_flag` (trailing-7d vs prior-7d clicks) | Label / proxy | the thing I'd rank pages by — a re-derived proxy for Lane 2's decline question, not the CSV's `is_declining_label` (that column doesn't exist on this table, and was itself rule-derived from `trend_pct`) |
| `content_id`, `client_id`, `report_date` | Context | pseudonymous IDs and dates — for grouping/joining/splitting only, never fed to a model |
| `ga4_*` engagement columns where `ga4_data_available = FALSE` | Excluded | zero-filled placeholders before a client's GA4 history starts — not real zero-engagement days |
| `gsc_avg_position = 0` (raw) | Excluded | means "no data," not rank zero — replaced with NULL before any averaging |

---

In [ ]:
# Supports the bucket table above: inspect real columns before trusting any name —
# search before assuming, per directing-your-ai-assistant.
con.sql(f"SELECT * FROM {FACT} LIMIT 5").df()

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

---

Grain was verified in section 1 (Query A). Two more claims, verified below: the slice's row
count and date span (Query B), and how much of the slice survives an honest `IS TRUE`
availability filter on `ga4_data_available` rather than trusting raw zeros (Query C).

---

In [ ]:
# Query B — slice row count and date span.
slice_shape = con.sql(f"""
    SELECT
        COUNT(*)                   AS n_rows,
        COUNT(DISTINCT content_id) AS n_content_items,
        MIN(report_date)           AS min_date,
        MAX(report_date)           AS max_date
    FROM {FACT}
""").df()
slice_shape

In [ ]:
# Query C — availability, filtered with IS TRUE (not a raw zero check).
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows,
        ROUND(100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_available
    FROM {FACT}
""").df()
availability

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

---

History depth differs wildly per client (`dim_clients.gsc_data_start` varies), so a single
global month like March 2026 represents different amounts of real history per client — some
clients have a full trailing history by March, others are only weeks into theirs. The query
below counts how many clients' `gsc_data_start` falls inside or after this month, which is the
concrete size of that gap. Separately, any decline proxy built later that needs a *trailing and
leading* window (see w03_feature_leakage_check.ipynb) can only be computed for the middle of the
month — the first and last week get dropped, which under-represents month-boundary days.

---

In [ ]:
# Supports the limitation above: how many clients have thin or no history by this month.
history_gap = con.sql(f"""
    SELECT
        COUNT(*) AS n_clients,
        SUM(CASE WHEN gsc_data_start >= DATE '2026-03-01' THEN 1 ELSE 0 END) AS clients_starting_in_or_after_march
    FROM {DIM_CLIENTS}
""").df()
history_gap

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.